# RecipeReel on AMD Instinct MI300X (ROCm)

This notebook demonstrates **actual AMD compute usage** for the AMD Developer Hackathon: ACT II
(Track 3). It runs RecipeReel's perception models — **Whisper-large-v3** (speech) and
**Qwen2.5-VL** (vision) — on the **AMD Instinct MI300X GPU via ROCm**, then turns a real
cooking video into a structured recipe.

Run this on the AMD Developer Cloud pod (notebooks.amd.com/hackathon).


## 1 · Verify the AMD GPU (ROCm)


In [ ]:
import torch
print('torch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())        # ROCm/HIP masquerades as CUDA
print('Device      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('ROCm/HIP    :', torch.version.hip)                 # set on AMD builds; CUDA is None
print('HBM total   : %.0f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9) if torch.cuda.is_available() else '')


In [ ]:
# Confirm it's an Instinct MI300X (gfx942)
!rocminfo | grep -m3 -i gfx || echo '(rocminfo not found — run on the AMD pod)'


## 2 · Install RecipeReel + GPU deps


In [ ]:
!git clone -q https://github.com/avinmaster/recipe-reel.git 2>/dev/null || echo 'already cloned'
%cd recipe-reel
!pip -q install -r requirements.txt -r requirements-gpu.txt
!apt-get -qq install -y ffmpeg >/dev/null 2>&1 || sudo apt-get -qq install -y ffmpeg
print('deps installed')


## 3 · Grab a real cooking video
Server-side YouTube download needs only Node (yt-dlp solves the JS/PO-token challenge) — no cookies.


In [ ]:
import subprocess, os
URL = 'https://www.youtube.com/shorts/rfYtSoCfmb4'   # a short cooking recipe
os.makedirs('demo', exist_ok=True)
subprocess.run(['yt-dlp','--js-runtimes','node','-f','best[height<=720]/best',
                '-o','demo/clip.%(ext)s', URL], check=False)
print('downloaded:', os.listdir('demo'))


## 4 · Run PERCEPTION on the MI300X
Whisper-large-v3 (ASR) and Qwen2.5-VL (vision) both load and run on the AMD GPU.
Synthesis uses Gemma-4 (set your free Google AI Studio key below, or point `SYNTHESIZER=amd`
at a vLLM-served Gemma on this same MI300X for a 100%-AMD run).


In [ ]:
import os, getpass, glob, torch, time
os.environ['TRANSCRIBER'] = 'local'    # Whisper-large-v3 on the MI300X
os.environ['VISION']      = 'local'    # Qwen2.5-VL on the MI300X
os.environ['SYNTHESIZER'] = 'fireworks'
os.environ['FIREWORKS_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
os.environ['SYNTH_MODEL'] = 'gemma-4-31b-it'
if not os.environ.get('FIREWORKS_API_KEY'):
    os.environ['FIREWORKS_API_KEY'] = getpass.getpass('Google AI Studio key (aistudio.google.com/apikey): ')

from app.models.enums import SourceType
from app.services.pipeline import run_pipeline
clip = glob.glob('demo/clip.*')[0]
t0 = time.time()
recipe = run_pipeline(source_type=SourceType.upload, source_ref=clip,
                      on_stage=lambda s,m: print(f'  [{s.value}] {m or ""}'))
print(f'\nPerception + synthesis in {time.time()-t0:.1f}s')
if torch.cuda.is_available():
    print('Peak GPU memory on MI300X: %.1f GB' % (torch.cuda.max_memory_allocated()/1e9))


## 5 · The structured recipe


In [ ]:
print('TITLE   :', recipe.title)
print('MODEL   :', recipe.model_used, '| vision:', recipe.processing.vision,
      '| device:', recipe.processing.device, '| gpu:', recipe.processing.gpu_name)
print('\nINGREDIENTS')
for i in recipe.ingredients:
    print('  -', i.quantity if i.quantity is not None else '', i.unit or '', i.name, f'[{i.source.value}]')
print('\nSTEPS')
for s in recipe.steps:
    print(f'  {s.number}. (@{s.start_time_seconds}s) {s.instruction}')


---
**AMD compute used:** Whisper-large-v3 and Qwen2.5-VL ran on the AMD Instinct MI300X (ROCm)
above — see the device / peak-GPU-memory output. `recipe.processing.device == 'cuda'` and
`gpu_name` name the Instinct card. This is RecipeReel's perception stage running on AMD hardware.
